# SF-3 — Reproducibility Notebook  ·  *The Quark Sector from 600-Cell Geometry* (v1.2)

**What this is.** A thin, executable wrapper over the canonical verifier
[`1500_verify_sf3_core.py`](1500_verify_sf3_core.py). It is *not* a re-derivation: it
**imports** the shipped script (so it cannot drift from it) and then walks each headline
number with the formula shown inline, **asserting** the value against the figure printed in
the SF-3 v1.2 paper. If any number ever drifts from the paper, a cell below **fails loudly**.

**The claim being reproduced.** The heavy-quark mass spectrum ($s,c,b,t$), the strong
coupling $\alpha_s$, the quark Koide phase $\theta_{\mathrm{quark}}$, and the three-generation
count all follow from a **single dimensionful calibration** — the electron mass $m_e$ — plus
600-cell geometry ($z=12$, $\varphi$) and $SU(3)$ colour ($C_F=4/3$). **Zero shape parameters
are fitted.** The charm mass is *derived*, not an input.

**How to run.** Top-to-bottom, standard library only (`math`). No data files, no network.


## 0 · Load the canonical verifier (single source of truth)

We load the shipped `.py` by file path (its name starts with a digit, so a normal `import`
won't do). Executing it runs the script's own self-test — its `[PASS]` table is printed below —
and hands us every computed value as a module attribute (`v.phi`, `v.me`, `v.M0`,
`v.alpha_s`, `v.theta`, `v.spectrum`, `v.PASS`, …). Every later cell sources its numbers from
`v`, so this notebook is physically incapable of diverging from the verifier.

In [1]:
import importlib.util, os, glob, math

# locate the canonical verifier next to this notebook (robust to run location)
cands = (glob.glob("1500_verify_sf3_core.py")
         or glob.glob(os.path.join("code", "1500_verify_sf3_core.py"))
         or glob.glob("**/1500_verify_sf3_core.py", recursive=True))
assert cands, "1500_verify_sf3_core.py not found next to the notebook"
spec = importlib.util.spec_from_file_location("sf3_verify", cands[0])
v = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v)   # <-- runs the canonical script (its self-test prints below)

print("\nloaded canonical verifier:", cands[0])
print("verifier self-test PASS:", v.PASS)

# small assertion helper: prints a PASS line and fails the cell if the number drifts
def show(label, got, want, tol, unit=""):
    ok = abs(got - want) <= tol
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:46s} {got:13.5f}{unit}   (paper {want}{unit})")
    assert ok, f"DRIFT: {label} = {got}{unit} but paper says {want}{unit} (tol {tol})"


phi = 1.6180339887
M0  = m_e z/phi = 3.7898 MeV   (paper: 3.79 MeV)
  [PASS] M0 anchor                                        3.7898 MeV  (target 3.79 MeV)

A. Quark mass spectrum  M_q = m_e (z/phi) V^(7/3)  [top x z*C_F]
  quark      V  mult     CPP(MeV)    PDG(MeV)    err%
  strange    4  1.00         96.3        93.4   +3.06
  [PASS] strange mass                                    96.2543 MeV  (target 96.3 MeV)
  charm     12  1.00       1249.4      1270.0   -1.62
  [PASS] charm mass                                    1249.4049 MeV  (target 1249.0 MeV)
  bottom    20  1.00       4114.8      4180.0   -1.56
  [PASS] bottom mass                                   4114.8146 MeV  (target 4115.0 MeV)
  top       30 16.00     169570.3    172760.0   -1.85
  [PASS] top mass                                    169570.3268 MeV  (target 169570.0 MeV)
  RMS error = 2.11%   (paper: 2.1%)
  [PASS] RMS residual                                     2.1101 %  (target 2.1 %)
  charm is DERIVED (single-m_

## 1 · The single calibration and the geometric inputs

$$M_0 \;=\; m_e\,\frac{z}{\varphi}\qquad
m_e = 0.511\ \text{MeV (the one calibration)},\quad z=12,\quad \varphi=\tfrac{1+\sqrt5}{2},\quad C_F=\tfrac43.$$

Paper value: $M_0 = 3.79$ MeV.

In [2]:
phi, me, z, CF = v.phi, v.me, v.z, v.CF      # primitives, straight from the verifier
M0 = me * z / phi                            # formula shown inline; value sourced from v's inputs
print(f"phi = {phi:.10f}")
print(f"m_e = {me} MeV  (single dimensionful calibration)")
print(f"z   = {z},   C_F = {CF:.4f}")
show("M_0 = m_e z/phi", M0, 3.79, 0.01, " MeV")
assert abs(M0 - v.M0) < 1e-12, "notebook M0 must equal verifier M0 (single source of truth)"
print("  M_0 matches the verifier's M_0 exactly.")


phi = 1.6180339887
m_e = 0.51099895 MeV  (single dimensionful calibration)
z   = 12,   C_F = 1.3333
  [PASS] M_0 = m_e z/phi                                      3.78978 MeV   (paper 3.79 MeV)
  M_0 matches the verifier's M_0 exactly.


## 2 · Zero-parameter quark mass spectrum

$$M_q \;=\; m_e\,\frac{z}{\varphi}\,V^{7/3}\,,\qquad V\in\{4,12,20,30\}=\{s,c,b,t\},
\qquad\text{top carries the relay multiplier } z\,C_F=16.$$

Paper: RMS residual **2.1 %** across four orders of magnitude. The **charm mass is derived**
($m_c = 1249$ MeV, $-1.6\%$) rather than calibrated — this is what restores the single-$m_e$
calibration.

In [3]:
errs = []
print(f"  {'quark':8s}{'V':>4s}{'mult':>6s}{'CPP(MeV)':>13s}{'PDG(MeV)':>12s}{'err%':>9s}")
for q, (V, mult, pdg, paper_cpp, paper_err) in v.spectrum.items():
    M = v.M0 * (V ** (7/3)) * mult            # the formula, on the verifier's M0
    e = (M - pdg) / pdg * 100
    errs.append(e)
    print(f"  {q:8s}{V:4d}{mult:6.2f}{M:13.1f}{pdg:12.1f}{e:+9.2f}")
    show(f"{q} mass", M, paper_cpp, max(2.0, 0.005*paper_cpp), " MeV")
rms = math.sqrt(sum(x*x for x in errs)/len(errs))
show("RMS residual", rms, 2.1, 0.1, " %")
mc = v.M0 * 12**(7/3)
print(f"\n  charm is DERIVED (not an input): m_c = {mc:.0f} MeV  ->  single-m_e calibration.")


  quark      V  mult     CPP(MeV)    PDG(MeV)     err%
  strange    4  1.00         96.3        93.4    +3.06
  [PASS] strange mass                                        96.25433 MeV   (paper 96.3 MeV)
  charm     12  1.00       1249.4      1270.0    -1.62
  [PASS] charm mass                                        1249.40485 MeV   (paper 1249.0 MeV)
  bottom    20  1.00       4114.8      4180.0    -1.56
  [PASS] bottom mass                                       4114.81458 MeV   (paper 4115.0 MeV)
  top       30 16.00     169570.3    172760.0    -1.85
  [PASS] top mass                                        169570.32682 MeV   (paper 169570.0 MeV)
  [PASS] RMS residual                                         2.11008 %   (paper 2.1 %)

  charm is DERIVED (not an input): m_c = 1249 MeV  ->  single-m_e calibration.


## 3 · Strong coupling and electroweak–strong complementarity

$$\alpha_s=\frac{5}{8\varphi}\approx0.386,\qquad \sin^2\theta_W=\frac{3}{8\varphi}\approx0.232,
\qquad \boxed{\;\sin^2\theta_W+\alpha_s=\frac1\varphi\;}\,,\qquad
\frac{\alpha_s}{\sin^2\theta_W}=\frac53=\frac{F}{E}=\frac{1200}{720}.$$

Edge- and face-mode fractions of one 600-cell spectral trace — a *structural correspondence*,
not a renormalization-group coupling (the paper is explicit on this).

In [4]:
alpha_s, sin2thW = v.alpha_s, v.sin2thW
show("alpha_s = 5/(8 phi)",          alpha_s,            5/(8*phi), 1e-9)
show("alpha_s value",                alpha_s,            0.386,     1e-3)
show("sin^2 theta_W = 3/(8 phi)",    sin2thW,            3/(8*phi), 1e-9)
show("complementarity sum = 1/phi",  alpha_s + sin2thW,  1/phi,     1e-9)
show("ratio = 5/3",                  alpha_s/sin2thW,    5/3,       1e-9)
show("topological F/E = 1200/720",   1200/720,           5/3,       1e-9)


  [PASS] alpha_s = 5/(8 phi)                                  0.38627   (paper 0.38627124296868426)
  [PASS] alpha_s value                                        0.38627   (paper 0.386)
  [PASS] sin^2 theta_W = 3/(8 phi)                            0.23176   (paper 0.23176274578121056)
  [PASS] complementarity sum = 1/phi                          0.61803   (paper 0.6180339887498948)
  [PASS] ratio = 5/3                                          1.66667   (paper 1.6666666666666667)
  [PASS] topological F/E = 1200/720                           1.66667   (paper 1.6666666666666667)


## 4 · Quark Koide phase

$$\varepsilon=\underbrace{-\frac{z\,\alpha_s}{z+1}}_{\varepsilon_S=-60/(104\varphi)}
+\underbrace{\frac{3}{52\varphi}}_{\varepsilon_{EW}}=-\frac{27}{52\varphi},\qquad
\cos\theta_{\mathrm{quark}}=-\frac23\!\left(1+\frac{\varepsilon}{2}\right),\qquad
\theta_{\mathrm{quark}}=124.04^\circ\ \ (\text{PDG }124.09^\circ).$$

**Proposition 5.1.** $\theta_{\mathrm{quark}}$ is built from $\{\alpha_s,\sin^2\theta_W,z\}$
only — no $m_c$, no mass amplitude — so it is independent of the mass route (a bookkeeping
separation, conditional on $\alpha_s$ being the structural value).

In [5]:
eps = v.eps
show("eps_S = -60/(104 phi)", v.eps_S, -60/(104*phi), 1e-9)
show("eps_EW = 3/(52 phi)",   v.eps_EW, 3/(52*phi),   1e-9)
show("eps = -27/(52 phi)",    eps,     -27/(52*phi),  1e-9)
costh = -(2/3)*(1 + eps/2)
theta = math.degrees(math.acos(costh))
print(f"  cos theta_quark = {costh:.5f}")
show("quark Koide phase", theta, 124.04, 0.02, " deg")
print("  Prop 5.1: phase depends on {alpha_s, sin^2 theta_W, z} only -> m_c-independent.")


  [PASS] eps_S = -60/(104 phi)                               -0.35656   (paper -0.35655807043263166)
  [PASS] eps_EW = 3/(52 phi)                                  0.03566   (paper 0.03565580704326317)
  [PASS] eps = -27/(52 phi)                                  -0.32090   (paper -0.3209022633893685)
  cos theta_quark = -0.55970
  [PASS] quark Koide phase                                  124.03500 deg   (paper 124.04 deg)
  Prop 5.1: phase depends on {alpha_s, sin^2 theta_W, z} only -> m_c-independent.


## 5 · Summary

Every headline number above was recomputed from the single calibration $m_e$ plus 600-cell
geometry and asserted against the SF-3 v1.2 paper. The cell below ties this notebook's result
to the canonical verifier's own global pass flag.

In [6]:
assert v.PASS, "canonical verifier reported a FAILED check"
print("Canonical verifier:           PASS")
print("Notebook headline assertions: PASS (all cells above)")
print()
print("=> ALL SF-3 v1.2 HEADLINE NUMBERS REPRODUCED FROM m_e + 600-CELL GEOMETRY.")
print("   Single calibration: m_e. Zero shape parameters. Charm derived. CKM open (OPEN-FP-3-CKM).")


Canonical verifier:           PASS
Notebook headline assertions: PASS (all cells above)

=> ALL SF-3 v1.2 HEADLINE NUMBERS REPRODUCED FROM m_e + 600-CELL GEOMETRY.
   Single calibration: m_e. Zero shape parameters. Charm derived. CKM open (OPEN-FP-3-CKM).
